In [25]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
from scipy.stats.mstats import gmean

import matplotlib.ticker as mtick 

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os
import json

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
country_list = pd.read_csv("C:\\heroku\\median-tariff\\data\\country-list-20.csv", header=None, names=["CTY_CODE"])

figfile = "C:\\github\\how-restrictive-us-trade\\figures\\"

texfile = "C:\\github\\how-restrictive-us-trade\\results.tex"

tabletex = "C:\\github\\how-restrictive-us-trade\\table.tex"

In [5]:
country_list.head(21)

,CTY_CODE
0,5700
1,2010
2,1220
3,5880
4,4280
5,5800
6,5520
7,5830
8,4190
9,5330


In [6]:
def make_tariff_country_date(dfcntry, date):
   
   bar = dfcntry[(dfcntry['time'] == date)].copy()

   bar["tariff"] = bar["CAL_DUT_MO"] / bar["CON_VAL_MO"]

   bar = bar[["I_COMMODITY", "CON_VAL_MO", "CAL_DUT_MO", "tariff", "CTY_NAME"]]

   return bar

In [7]:
def make_good_country_year(dfcntry, year):
    """
    Process country data for a specific year and calculate aggregate metrics.
    
    Parameters:
    -----------
    dfcntry : pandas.DataFrame
        DataFrame containing country trade data with columns:
        - time: datetime
        - I_COMMODITY: commodity code
        - CON_VAL_MO: consumption value
        - CAL_DUT_MO: calculated duty
        - CTY_NAME: country name
    year : str
        Year to filter data (e.g., "2024")
    
    Returns:
    --------
    pandas.DataFrame
        Aggregated data by commodity for the specified year
    """
    # Filter to specific year
    df_year = dfcntry[dfcntry['time']== year]
    
    # Group by commodity and aggregate
    grp = df_year.groupby("I_COMMODITY")
    
    result = grp.agg({
        "CON_VAL_MO": "sum",
        "CAL_DUT_MO": "sum",
        "CTY_NAME": "first"})
    
    
    return result

In [8]:
country_list.head()

,CTY_CODE
0,5700
1,2010
2,1220
3,5880
4,4280


In [9]:
def process_countries_for_date(country_list, target_date, weight_year="2024", selected_country=None):
    """
    Process multiple countries and create combined dataset with weights.
    
    Parameters:
    -----------
    country_list : pandas.DataFrame or list
        DataFrame with CTY_CODE column or list of country codes
    target_date : str
        Date to extract tariff data for (e.g., "2025-08")
    weight_year : str, optional
        Year to use for calculating weights (default: "2024")
    selected_country : str, optional
        Country name to filter results (e.g., "CANADA"). If None, returns all countries (default: None)
    
    Returns:
    --------
    pandas.DataFrame
        Combined dataframe with tariff data and weights for all countries (or selected country)
    """
    # Handle both DataFrame and list inputs
    if isinstance(country_list, pd.DataFrame):
        countries = country_list.CTY_CODE
    else:
        countries = country_list
    
    all_results = []
    tariff_results = []
    
    for country in countries:
        
        file = f"C:\\heroku\\median-tariff\\data\\imports-hs10\\{country}data-current.parquet"
        
        # Load country data
        dfcntry = pq.read_table(file).to_pandas()
        
        dfcntry["CAL_DUT_MO"] = dfcntry["CAL_DUT_MO"].astype(float)

        dfcntry["CON_VAL_MO"] = dfcntry["CON_VAL_MO"].astype(float)

        dfcntry.time = pd.to_datetime(dfcntry.time, format="%Y-%m")

        dfcntry["applied_tariff"] = dfcntry["CAL_DUT_MO"] / dfcntry["CON_VAL_MO"]
            
        # Calculate metrics for weight year
        results_df = make_good_country_year(dfcntry, weight_year)

        results_df.reset_index(inplace=True)
        
        # Get tariff data for target date
        tariff_df = make_tariff_country_date(dfcntry, target_date)

        tariff_df.reset_index(drop=True, inplace=True)
        
        # Add to lists
        all_results.append(results_df)

        tariff_results.append(tariff_df)
        
        #print(f"Processed {country}: {results_df['CTY_NAME'].iloc[0]}")
    
    # Combine all results into one dataframe
    combined_results = pd.concat(all_results)
    combined_results.reset_index(drop=True, inplace=True)
    
    combined_tariff_results = pd.concat(tariff_results)
    combined_tariff_results.reset_index(drop=True, inplace=True)
    
    # Calculate weights
    combined_results["weights"] = combined_results["CON_VAL_MO"] / combined_results["CON_VAL_MO"].sum()
    
    # Merge weights with tariff data
    bigdf = pd.merge(
        combined_results[["I_COMMODITY", "CTY_NAME", "weights"]], 
        combined_tariff_results, 
        how="left", 
        on=["I_COMMODITY", "CTY_NAME"]
    )
    
    # Filter to selected country if specified
    if selected_country is not None:
        bigdf = bigdf[bigdf["CTY_NAME"] == selected_country].copy()
        # Renormalize weights to sum to 1 for the selected country
        bigdf["weights"] = bigdf["weights"] / bigdf["weights"].sum()
    
    return bigdf

In [10]:
def create_tariff_metrics_by_date_country(country_list, date_list, weight_year="2024"):
    """
    Calculate tariff metrics for multiple dates and countries.
    
    Parameters:
    -----------
    country_list : pandas.DataFrame or list
        DataFrame with CTY_CODE column or list of country codes
    date_list : list
        List of date strings to process (e.g., ["2024-08", "2025-02", "2025-08"])
    weight_year : str, optional
        Year to use for calculating weights (default: "2024")
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with columns: date, CTY_NAME, sqrtariff, meanweighted, simplemean
    """
    results = []
    
    for date in date_list:
        print(f"Processing date: {date}")
        
        # Get data for this date (all countries)
        bigdf = process_countries_for_date(country_list, date, weight_year=weight_year)
        
        # Calculate metrics for ALL COUNTRIES combined
        sqrtariff_all = ((bigdf["tariff"]**2 * bigdf["weights"]).sum())**0.5
        meanweighted_all = (bigdf["tariff"] * bigdf["weights"]).sum()
        simplemean_all = bigdf['CAL_DUT_MO'].sum() / bigdf["CON_VAL_MO"].sum()
        
        # Store ALL COUNTRIES results
        results.append({
            'date': date,
            'CTY_NAME': 'ALL COUNTRIES',
            'sqrtariff': sqrtariff_all,
            'meanweighted': meanweighted_all,
            'simplemean': simplemean_all
        })
        
        # Get unique countries in this dataset
        countries = bigdf['CTY_NAME'].unique()
        
        for country in countries:
            # Filter to this country
            country_df = bigdf[bigdf['CTY_NAME'] == country].copy()
            
            # Renormalize weights to sum to 1 for this country
            country_df["weights"] = country_df["weights"] / country_df["weights"].sum()
            
            # Calculate metrics
            sqrtariff = ((country_df["tariff"]**2 * country_df["weights"]).sum())**0.5
            meanweighted = (country_df["tariff"] * country_df["weights"]).sum()
            simplemean = country_df['CAL_DUT_MO'].sum() / country_df["CON_VAL_MO"].sum()
            
            # Store results
            results.append({
                'date': date,
                'CTY_NAME': country,
                'sqrtariff': sqrtariff,
                'meanweighted': meanweighted,
                'simplemean': simplemean
            })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

In [ ]:
# Country to flag URL mapping
country_flags = {
    'China': 'https://flagcdn.com/w40/cn.png',
    'Japan': 'https://flagcdn.com/w40/jp.png',
    'Mexico': 'https://flagcdn.com/w40/mx.png',
    'Canada': 'https://flagcdn.com/w40/ca.png',
    'Vietnam': 'https://flagcdn.com/w40/vn.png',
    'Korea, South': 'https://flagcdn.com/w40/kr.png',
    'India': 'https://flagcdn.com/w40/in.png',
    'Taiwan': 'https://flagcdn.com/w40/tw.png',
    'Germany': 'https://flagcdn.com/w40/de.png',
    'United Kingdom': 'https://flagcdn.com/w40/gb.png',
    'France': 'https://flagcdn.com/w40/fr.png',
    'Italy': 'https://flagcdn.com/w40/it.png',
    'Ireland': 'https://flagcdn.com/w40/ie.png',
    'Switzerland': 'https://flagcdn.com/w40/ch.png',
    'Thailand': 'https://flagcdn.com/w40/th.png',
    'Netherlands': 'https://flagcdn.com/w40/nl.png',
    'Brazil': 'https://flagcdn.com/w40/br.png',
    'Belgium': 'https://flagcdn.com/w40/be.png',
    'Singapore': 'https://flagcdn.com/w40/sg.png',
    'Indonesia': 'https://flagcdn.com/w40/id.png',
    'Malaysia': 'https://flagcdn.com/w40/my.png',
}

# Country to color mapping (using national/flag colors)
country_colors = {
    'China': '#DE2910',
    'Japan': '#BC002D',
    'Mexico': '#006847',
    'Canada': '#FF0000',
    'Vietnam': '#DA251D',
    'Korea, South': '#003478',
    'India': '#FF9933',
    'Taiwan': '#000095',
    'Germany': '#FFCE00',
    'United Kingdom': '#012169',
    'France': '#0055A4',
    'Italy': '#009246',
    'Ireland': '#169B62',
    'Switzerland': '#FF0000',
    'Thailand': '#2D2A4A',
    'Netherlands': '#FF6600',
    'Brazil': '#009B3A',
    'Belgium': '#FDDA24',
    'Singapore': '#EE2737',
    'Indonesia': '#FF0000',
    'Malaysia': '#CC0001',
}

foo['flag'] = foo['country'].map(country_flags)
foo['color'] = foo['country'].map(country_colors)

# Format amount as currency string
foo['amount'] = foo['amount'].apply(lambda x: f"${x:,.0f}")

# Format rate as percentage string
foo['rate'] = foo['rate'].apply(lambda x: f"{x:.0f}%")

# Reorder columns
foo = foo[['country', 'flag', 'amount', 'color', 'rate']]

# Convert to list of dictionaries
tariffs = foo.to_dict('records')

# Write to JSON file
with open("tariffs.json", "w") as f:
    json.dump(tariffs, f, indent=2)



print("tariffs.json created successfully!")

Processing date: 2025-09
tariffs.json created successfully!
